# Aula 2 — Receitas aplicadas em cada pixel

**Ideia central:** Toda transformação é uma receita repetida em cada pixel. Se a receita olha só o pixel, é operação de ponto. Se olha os vizinhos, é filtro. Se usa um carimbo, é morfologia.

**Ementa:** Transformações de intensidade • Filtragem espacial • Convolução • Morfologia matemática

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from skimage import io, color, exposure, filters, morphology, util
from scipy import ndimage as ndi

## 1. Carregar imagem

In [ ]:
# Use uma imagem local ou faça upload
# from google.colab import files
# nome = list(files.upload().keys())[0]
# img = io.imread(nome)[..., :3]

# Exemplo com imagem de teste
from skimage import data
img = data.astronaut()  # Substitua por sua imagem
cinza = color.rgb2gray(img)  # float entre 0 e 1

plt.imshow(img)
plt.title("Imagem original")
plt.axis("off")
plt.show()

## 2. Transformações de intensidade

In [ ]:
eq = exposure.equalize_hist(cinza)         # equalização global
clahe = exposure.equalize_adapthist(cinza) # equalização local (CLAHE)
gama = exposure.adjust_gamma(cinza, gamma=0.5)  # clareia os escuros

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(cinza, cmap="gray")
ax[0].set_title("Original (cinza)")
ax[1].imshow(eq, cmap="gray")
ax[1].set_title("Equalização global")
ax[2].imshow(clahe, cmap="gray")
ax[2].set_title("CLAHE")
ax[3].imshow(gama, cmap="gray")
ax[3].set_title("Gamma = 0.5")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 3. Filtragem espacial

In [ ]:
# Adiciona ruído sal e pimenta
ruidosa = util.random_noise(cinza, mode="s&p", amount=0.06)

# Filtros
media = ndi.uniform_filter(ruidosa, size=3)
mediana = ndi.median_filter(ruidosa, size=3)

# Detecção de bordas com Sobel
sobel_x = ndi.convolve(cinza, np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]))

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(ruidosa, cmap="gray")
ax[0].set_title("Sal e pimenta")
ax[1].imshow(media, cmap="gray")
ax[1].set_title("Média 3x3")
ax[2].imshow(mediana, cmap="gray")
ax[2].set_title("Mediana 3x3")
ax[3].imshow(np.abs(sobel_x), cmap="gray")
ax[3].set_title("|Sobel x|")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

## 4. Morfologia matemática

Operações sobre máscara binária de planta (limiar em a*, verde = a* negativo).

In [ ]:
# Segmentação simples pelo canal a* (verde tem a* negativo)
a = color.rgb2lab(img)[..., 1]
mascara = a < -8

# Operações morfológicas
limpa = morphology.binary_opening(mascara, morphology.disk(1))
sem_buraco = ndi.binary_fill_holes(
    morphology.binary_closing(limpa, morphology.disk(3))
)

fig, ax = plt.subplots(1, 4, figsize=(16, 4))
ax[0].imshow(mascara, cmap="gray")
ax[0].set_title("Máscara (a* < -8)")
ax[1].imshow(limpa, cmap="gray")
ax[1].set_title("Abertura (disco r=1)")
ax[2].imshow(sem_buraco, cmap="gray")
ax[2].set_title("Fechamento + preencher")
ax[3].imshow(img)
ax[3].imshow(sem_buraco, alpha=0.3, cmap="Greens")
ax[3].set_title("Sobreposição")
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()

---

## Desafios

### Aquecimento
Monte um kernel que detecte só bordas na diagonal. Quais pesos você usou? A soma dá quanto?

### Mão na massa
Implemente a erosão binária em NumPy puro, sem scikit-image, usando `np.roll` ou fatiamento.

### Investigação
Calcule a cobertura verde antes e depois de abertura com disco de raio 1, 2 e 3. Faça um gráfico raio × cobertura.

In [ ]:
# Espaço para os desafios


---

## Leitura recomendada

- GONZALEZ, R. C.; WOODS, R. E. *Processamento digital de imagens*. 3. ed. Cap. 3 e 9.
- PEDRINI, H.; SCHWARTZ, W. R. *Análise de imagens digitais*. Thomson, 2008.